# Anomaly Detection — Cycling 

1. **Rolling-median pass** — flags any daily count above 4× its local 30-day median.
2. **Isolation Forest** — per-site multivariate detector that catches subtler faults using weather context (e.g. a count that's only suspicious because it happened during heavy rain on a Tuesday in January).

In [1]:
import requests
import pandas as pd
import numpy as np
from io import StringIO
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Download the cycling count data

In [2]:
BASE_URL = "https://opendata.apps.mow.vlaanderen.be/fietstellingen"

COUNT_COLS  = ['site ID', 'richting', 'type', 'van', 'tot', 'aantal']
COUNT_DTYPE = {
    'site ID':  'int32',
    'richting': 'category',
    'type':     'category',
    'van':      'string',
    'tot':      'string',
    'aantal':   'float32',
}

def download_counts(year: int, month: int, save_dir: str = "data") -> pd.DataFrame:
    Path(save_dir).mkdir(exist_ok=True)
    filename   = f"data-{year}-{month:02d}.csv"
    local_path = Path(save_dir) / filename

    if local_path.exists():
        print(f"  {filename} already cached locally")
    else:
        url = f"{BASE_URL}/{filename}"
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        local_path.write_bytes(resp.content)
        print(f"  Downloaded {filename} ({len(resp.content)/1024:.0f} KB)")

    df = pd.read_csv(
        local_path,
        sep=',',
        header=None,
        names=COUNT_COLS,
        dtype=COUNT_DTYPE,
        low_memory=False,
    )
    df = df[df['type'] == 'FIETSERS']
    df['ds'] = pd.to_datetime(df['van'], errors='coerce').dt.normalize()
    daily = (
        df.groupby(['site ID', 'ds'], observed=True)['aantal']
          .sum()
          .reset_index()
    )
    return daily

MONTHS_TO_DOWNLOAD = (
    [(2024, m) for m in range(1, 13)]
    + [(2025, m) for m in range(1, 13)]
    + [(2026, m) for m in range(1, 5)]
)

print("Downloading cycling count data...")
daily_frames = []
for year, month in MONTHS_TO_DOWNLOAD:
    try:
        daily_frames.append(download_counts(year, month))
    except Exception as e:
        print(f"  Skipping {year}-{month:02d}: {e}")

raw_daily = pd.concat(daily_frames, ignore_index=True)
print(f"\nTotal daily rows: {len(raw_daily):,}")
print(raw_daily.head(3))

  data-2024-01.csv already cached locally
  data-2024-02.csv already cached locally
  data-2024-03.csv already cached locally
  data-2024-04.csv already cached locally
  data-2024-05.csv already cached locally
  data-2024-06.csv already cached locally
  data-2024-07.csv already cached locally
  data-2024-08.csv already cached locally
  data-2024-09.csv already cached locally
  data-2024-10.csv already cached locally
  data-2024-11.csv already cached locally
  data-2024-12.csv already cached locally
  data-2025-01.csv already cached locally
  data-2025-02.csv already cached locally
  data-2025-03.csv already cached locally
  data-2025-04.csv already cached locally
  data-2025-05.csv already cached locally
  data-2025-06.csv already cached locally
  data-2025-07.csv already cached locally
  data-2025-08.csv already cached locally
  data-2025-09.csv already cached locally
  data-2025-10.csv already cached locally
  data-2025-11.csv already cached locally
  data-2025-12.csv already cached 

## Build a continuous daily series

Reindex each site to a gap-free daily calendar and fill short gaps (linear interpolation up to 3 days, then same-weekday fallback). The `fill_gaps` helper is reused after the rolling-median pass NaNs out flagged points.

In [ ]:
daily = (
    raw_daily
    .groupby(['site ID', 'ds'], observed=True)['aantal']
    .sum()
    .reset_index()
    .rename(columns={'site ID': 'unique_id', 'aantal': 'y'})
)

def make_full_index(g):
    return pd.DataFrame({
        'unique_id': g['unique_id'].iloc[0],
        'ds': pd.date_range(g['ds'].min(), g['ds'].max(), freq='D')
    })

full_idx = pd.concat([make_full_index(g) for _, g in daily.groupby('unique_id')])
daily    = full_idx.merge(daily, on=['unique_id', 'ds'], how='left')

def fill_gaps(s):
    s = s.interpolate('linear', limit=3)
    mask = s.isna()
    s[mask] = s.shift(7).combine_first(s.shift(-7))[mask]
    return s

daily['y'] = daily.groupby('unique_id')['y'].transform(fill_gaps)

print(f"Daily dataframe shape: {daily.shape}")
print(daily.head(3))

## Layer 1 — Rolling-median outlier detection

Flag any count exceeding 4× the local 30-day median as suspect. The flag is kept as a `rolling_outlier` column, then the suspect value is NaN'd and the gap-fill re-run so the cleaned series feeds Layer 2. `y_observed` preserves the value as it was before cleaning, for the final report.

In [ ]:
# Preserve the observed value before we NaN-and-refill flagged points
daily['y_observed'] = daily['y']

rolling_med = daily.groupby('unique_id')['y'].transform(
    lambda x: x.rolling(30, min_periods=7, center=True).median()
)
suspect = (daily['y'] > 4 * rolling_med) & rolling_med.notna()
daily['rolling_outlier'] = suspect.astype('int8')
print(f"Rolling-median outliers flagged: {int(suspect.sum())}")
daily.loc[suspect, 'y'] = np.nan

daily['y'] = daily.groupby('unique_id')['y'].transform(fill_gaps)
print(f"Remaining NaN in y: {int(daily['y'].isna().sum())}")

## Attach weather covariates

Weather is downloaded by the companion notebook and pushed to Hugging Face. Hourly readings are aggregated into the daily features the Isolation Forest uses as context. The weather `site_id` matches the cycling `unique_id`, so the join is direct.

In [ ]:
from huggingface_hub import hf_hub_download

weather_hourly = pd.read_csv(hf_hub_download(
    repo_id="Zelal-iab/MDA_Project_Data",
    filename="weather_data_2024-2026.csv",
    repo_type="dataset",
))
weather_hourly['time'] = pd.to_datetime(weather_hourly['time'])
weather_hourly['ds']   = weather_hourly['time'].dt.normalize()

weather_daily = (
    weather_hourly
    .groupby(['site_id', 'ds'])
    .agg(
        temperature_2m_max = ('temperature_2m',    'max'),
        precipitation_sum  = ('precipitation',     'sum'),
        wind_speed_10m_max = ('wind_speed_10m',    'max'),
        sunshine_duration  = ('sunshine_duration', 'sum'),
    )
    .reset_index()
    .rename(columns={'site_id': 'unique_id'})
)

daily['unique_id']         = daily['unique_id'].astype('int64')
weather_daily['unique_id'] = weather_daily['unique_id'].astype('int64')

daily = daily.merge(weather_daily, on=['unique_id', 'ds'], how='left')
coverage = 1 - daily['temperature_2m_max'].isna().mean()
print(f"Daily shape: {daily.shape}")
print(f"Weather coverage: {coverage * 100:.1f}%")
print(daily.head(3))

## Layer 2 — Isolation Forest

Per-site multivariate anomaly detection on top of the rolling-median check. Catches subtler patterns by combining deviation-from-median, week-over-week and 4-week change, day-of-week / month seasonality, and rain/sun context. Produces the `sensor_fault` binary flag.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

def build_iso_features(g):
    g = g.copy()
    g['dow']           = g['ds'].dt.dayofweek
    g['month']         = g['ds'].dt.month
    g['rolling_med']   = g['y'].rolling(30, min_periods=7, center=True).median()
    g['deviation']     = g['y'] - g['rolling_med']
    g['vs_last_week']  = g['y'] - g['y'].shift(7)
    g['vs_4weeks_ago'] = g['y'] - g['y'].shift(28)
    g['rain_spike']    = (g['precipitation_sum']   > 10).astype(int)
    g['sun_zero']      = (g['sunshine_duration'] == 0).astype(int)
    return g

iso_features = ['deviation', 'vs_last_week', 'vs_4weeks_ago',
                'dow', 'month', 'rain_spike', 'sun_zero']

flag_frames = []
for uid, group in daily.groupby('unique_id'):
    g = build_iso_features(group).dropna(subset=iso_features)
    if len(g) < 60:
        continue
    X = StandardScaler().fit_transform(g[iso_features].values)
    clf = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
    g['sensor_fault'] = (clf.fit_predict(X) == -1).astype(int)
    flag_frames.append(g[['unique_id', 'ds', 'sensor_fault']])

flags = pd.concat(flag_frames, ignore_index=True)
daily = daily.merge(flags, on=['unique_id', 'ds'], how='left')
daily['sensor_fault'] = daily['sensor_fault'].fillna(0).astype('int8')

print(f"Sites scored: {flags['unique_id'].nunique()}")
print(f"Sensor-fault flags: {int(daily['sensor_fault'].sum())} of {len(daily):,}")

## Combine layers & export

A site-day is anomalous if either layer flagged it. Exported to `anomalies.csv` with `y_observed` (the value as originally recorded) and a column per layer.

In [ ]:
anomalies = (
    daily[(daily['rolling_outlier'] == 1) | (daily['sensor_fault'] == 1)]
    [['unique_id', 'ds', 'y_observed', 'rolling_outlier', 'sensor_fault']]
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

print(f"Total anomalous site-days: {len(anomalies):,}")
print(f"  Rolling-median outliers : {int(daily['rolling_outlier'].sum())}")
print(f"  Isolation-Forest faults : {int(daily['sensor_fault'].sum())}")
print(f"  Flagged by both         : {int(((daily['rolling_outlier'] == 1) & (daily['sensor_fault'] == 1)).sum())}")

anomalies.to_csv('anomalies.csv', index=False)
print("\nSaved -> anomalies.csv")
anomalies.head(10)